# 🎯 使用最优参数的视频分割（改进版 - 匹配实际边界）

**最优参数**: ws=23, thr=0.80, sigma=100

**✅ Activity层分割点**: 02:29（非常接近目标02:28！）

**✅ 使用与参数倒推测试完全一致的方法**:
1. 片段边界：`prev = b + 1`（边界后从下一帧开始）
2. 特征提取：`features[start:end+1]`（闭区间）
3. **不归一化** atomic_features
4. SCM构建：直接矩阵乘法 `atomic_features @ atomic_features.T`
5. **匹配实际边界帧**（而非原子中心点）

---
## 📦 环境配置

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import savgol_filter
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境配置完成")

✅ 环境配置完成


---
## 🔧 定义分割算法

In [2]:
OPTIMAL_PARAMS = {'ws': 23, 'thr': 0.80, 'sigma': 100}
FPS = 10
video_name = 'C10390_rgb'

def format_time(seconds):
    return f"{int(seconds//60):02d}:{int(seconds%60):02d}"

def compute_window_similarity(window_a, window_b, method='cosine'):
    if method == 'cosine':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        na, nb = np.linalg.norm(mean_a), np.linalg.norm(mean_b)
        if na == 0 or nb == 0:
            return 0.0
        return float(np.dot(mean_a, mean_b) / (na * nb))
    return 0.0

def sw_asymmetric(feats, window_size=10, step_reset=1, threshold=0.8,
                  similarity_method='cosine', min_segment_frames=10):
    T = feats.shape[0]
    right_size = int(window_size)
    left_init = int(window_size)
    grow = 1
    max_left = right_size * 3
    boundaries, similarities, positions = [], [], []
    pos, left_len = 0, left_init
    while pos + left_len + right_size <= T:
        left_start = pos
        left_end = pos + left_len
        right_end = left_end + right_size
        a = feats[left_start:left_end]
        b = feats[left_end:right_end]
        sim = compute_window_similarity(a, b, similarity_method)
        similarities.append(float(sim))
        positions.append(left_end)
        if sim < threshold:
            mid = left_end
            if (not boundaries) or (mid - boundaries[-1] >= min_segment_frames):
                boundaries.append(int(mid))
                pos = mid
                left_len = left_init
            else:
                left_len = min(left_len + grow, max_left)
                if (pos + left_len + right_size > T) or (left_len >= max_left):
                    pos += step_reset
                    left_len = left_init
        else:
            left_len = min(left_len + grow, max_left)
            if (pos + left_len + right_size > T) or (left_len >= max_left):
                pos += step_reset
                left_len = left_init
    return boundaries, similarities, positions

def traditional_diff_peaks(feats_scaled, fps):
    div_boundaries = []
    try:
        diffs = feats_scaled[1:] - feats_scaled[:-1]
        l2 = np.linalg.norm(diffs, axis=1)
        if len(l2) >= 5:
            win = min(7, len(l2) // 4)
            win += 1 if win % 2 == 0 else 0
            if win >= 3:
                l2_s = savgol_filter(l2, win, 1, mode='mirror')
                thr = np.percentile(l2_s, 85)
                cand = np.where(l2_s > thr)[0]
                gap = max(1, int(0.3 * fps))
                for c in cand:
                    if not div_boundaries or c - div_boundaries[-1] >= gap:
                        div_boundaries.append(int(c))
    except Exception:
        pass
    return div_boundaries

def refine_boundary(b, feats, fps, radius=None):
    if radius is None:
        radius = max(2, int(0.3 * fps))
    L = feats.shape[0]
    start = max(2, b - radius)
    end = min(L - 2, b + radius)
    diffs = feats[1:] - feats[:-1]
    l2 = np.linalg.norm(diffs, axis=1)
    l2_smooth = np.convolve(l2, np.ones(3)/3, mode='same') if len(l2) >= 3 else l2
    idx = np.argmax(l2_smooth[start:end+1])
    return int(start + idx)

def post_fuse_and_refine(boundaries_a, boundaries_b, feats_scaled, fps):
    MIN_DIST = max(1, int(0.3 * fps))
    merged = sorted(set(boundaries_a + boundaries_b))
    final = []
    for b in merged:
        if not final or b - final[-1] >= MIN_DIST:
            final.append(int(b))
    refined = []
    for b in final:
        rb = refine_boundary(b, feats_scaled, fps)
        if not refined or rb - refined[-1] >= MIN_DIST:
            refined.append(rb)
    return refined

print("✅ 算法定义完成")

✅ 算法定义完成


---
## 🔄 运行完整分割流程（参数倒推测试方法）

In [3]:
print("="*70)
print("🔄 使用最优参数运行分割（参数倒推测试方法）")
print("="*70)
print(f"📋 参数: ws={OPTIMAL_PARAMS['ws']}, thr={OPTIMAL_PARAMS['thr']:.2f}, sigma={OPTIMAL_PARAMS['sigma']}")
print()

# 加载特征
features = np.load('features.npy')
T = features.shape[0]
print(f"✓ 特征: {features.shape}, 时长: {T/FPS:.1f}秒")

# 标准化
scaler = StandardScaler()
feats_scaled = scaler.fit_transform(features)

# 不对称滑动窗口
min_seg = max(FPS, OPTIMAL_PARAMS['ws'])
sw_boundaries, _, _ = sw_asymmetric(
    feats_scaled, window_size=OPTIMAL_PARAMS['ws'], step_reset=1,
    threshold=OPTIMAL_PARAMS['thr'], similarity_method='cosine', min_segment_frames=min_seg
)
print(f"✓ 滑动窗口: {len(sw_boundaries)} 个边界")

# 差分峰值
div_boundaries = traditional_diff_peaks(feats_scaled, FPS)
print(f"✓ 差分峰值: {len(div_boundaries)} 个边界")

# 融合精修
boundaries = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled, FPS)
print(f"✓ 融合精修: {len(boundaries)} 个边界")
print()

# ✅ 关键：构建片段（与参数倒推测试第1234-1240行一致）
segs = []
prev = 0
for b in boundaries:
    segs.append((prev, b))
    prev = b + 1  # ✅ 边界后从下一帧开始
segs.append((prev, T - 1))  # ✅ 最后一个片段到T-1
K = len(segs)
print(f"✓ 原子单元: {K}个")
print()

# ✅ 关键：构建原子特征（与参数倒推测试第1252-1260行一致）
atomic_features = []
atomic_centers = []
for start, end in segs:
    seg_feat = features[start:end+1].mean(axis=0)  # ✅ 闭区间
    atomic_features.append(seg_feat)  # ✅ 不归一化
    atomic_centers.append((start + end) / 2 / FPS)
atomic_features = np.array(atomic_features)
atomic_centers = np.array(atomic_centers)
print(f"✓ 原子特征: {atomic_features.shape}（未归一化）")
print()

# ✅ 关键：构建SCM（与参数倒推测试第1263-1267行一致）
sigma = OPTIMAL_PARAMS['sigma']
cos_sim = atomic_features @ atomic_features.T  # ✅ 直接矩阵乘法
time_diff = np.abs(atomic_centers[:, None] - atomic_centers[None, :])
temporal_kernel = np.exp(-time_diff**2 / (2 * sigma**2))
SCM = cos_sim * temporal_kernel
print(f"✓ SCM: {SCM.shape}, 范围: [{SCM.min():.4f}, {SCM.max():.4f}]")
print()

# 层次聚类（与参数倒推测试第1269-1273行一致）
dist_matrix = 1 - SCM
np.fill_diagonal(dist_matrix, 0)
condensed_dist = dist_matrix[np.triu_indices(K, k=1)]
Z = linkage(condensed_dist, method='ward')
print(f"✓ 聚类完成: {Z.shape}")
print()

# 多粒度分割
def labels_to_segments(labels, atomic_segs):
    segments = []
    K = len(labels)
    event_boundaries = [0]
    for i in range(1, K):
        if labels[i] != labels[i-1]:
            event_boundaries.append(i)
    event_boundaries.append(K)
    for i in range(len(event_boundaries) - 1):
        start_idx = event_boundaries[i]
        end_idx = event_boundaries[i+1] - 1
        start_frame = atomic_segs[start_idx][0]
        end_frame = atomic_segs[end_idx][1]
        segments.append((start_frame, end_frame))
    return segments

hierarchical_segs = {}
hierarchical_segs['activity'] = labels_to_segments(fcluster(Z, t=2, criterion='maxclust'), segs)
hierarchical_segs['side'] = labels_to_segments(fcluster(Z, t=4, criterion='maxclust'), segs)
hierarchical_segs['component'] = labels_to_segments(fcluster(Z, t=8, criterion='maxclust'), segs)
hierarchical_segs['atomic'] = segs

print("="*70)
print("✅ 分割完成！")
print("="*70)
print(f"层次结构: 2 → 4 → 8 → {K}")
print()

🔄 使用最优参数运行分割（参数倒推测试方法）
📋 参数: ws=23, thr=0.80, sigma=100

✓ 特征: (2785, 512), 时长: 278.5秒
✓ 滑动窗口: 83 个边界
✓ 差分峰值: 159 个边界
✓ 融合精修: 141 个边界

✓ 原子单元: 142个

✓ 原子特征: (142, 512)（未归一化）

✓ SCM: (142, 142), 范围: [0.0219, 0.9937]

✓ 聚类完成: (141, 4)

✅ 分割完成！
层次结构: 2 → 4 → 8 → 142



---
## 📋 完整时间线输出

In [4]:
print("="*70)
print("📋 完整分割时间线")
print("="*70)
print()

for level_name, icon, title in zip(
    ['activity', 'side', 'component', 'atomic'],
    ['🔴', '🟠', '🟢', '🔵'],
    ['ACTIVITY', 'SIDE', 'COMPONENT', 'ATOMIC']
):
    segs_level = hierarchical_segs[level_name]
    num_events = len(segs_level)
    print(f"{icon} {title}层 - {num_events} 个事件")
    print("="*70)
    for i, (start_frame, end_frame) in enumerate(segs_level):
        start_time = start_frame / FPS
        end_time = end_frame / FPS
        duration_event = end_time - start_time
        if level_name != 'atomic' or num_events <= 20:
            print(f"  Event {i+1}: {format_time(start_time)} - {format_time(end_time)}  "
                  f"(时长: {duration_event:6.1f}秒, {int(duration_event/60):2d}分{int(duration_event%60):02d}秒)")
        elif i == 20:
            print(f"  ... (省略剩余 {num_events-20} 个原子事件)")
    if num_events > 1 and level_name != 'atomic':
        boundary_times = [seg[1] / FPS for seg in segs_level[:-1]]
        print(f"\n  🔹 分割点 ({len(boundary_times)}个): {', '.join([format_time(t) for t in boundary_times])}")
    print("\n")

print("="*70)
print("✅ 完成！使用与参数倒推测试完全一致的方法")
print("="*70)
print()
print("💡 关键差异:")
print("   • 片段边界: prev = b + 1（边界后从下一帧开始）")
print("   • 特征提取: features[start:end+1]（闭区间）")
print("   • 不归一化 atomic_features")
print("   • SCM: 直接矩阵乘法")
print()

📋 完整分割时间线

🔴 ACTIVITY层 - 2 个事件
  Event 1: 00:00 - 02:29  (时长:  149.5秒,  2分29秒)
  Event 2: 02:29 - 04:38  (时长:  128.8秒,  2分08秒)

  🔹 分割点 (1个): 02:29


🟠 SIDE层 - 4 个事件
  Event 1: 00:00 - 01:20  (时长:   80.2秒,  1分20秒)
  Event 2: 01:20 - 02:29  (时长:   69.2秒,  1分09秒)
  Event 3: 02:29 - 03:35  (时长:   65.4秒,  1分05秒)
  Event 4: 03:35 - 04:38  (时长:   63.3秒,  1分03秒)

  🔹 分割点 (3个): 01:20, 02:29, 03:35


🟢 COMPONENT层 - 8 个事件
  Event 1: 00:00 - 00:31  (时长:   31.5秒,  0分31秒)
  Event 2: 00:31 - 00:52  (时长:   20.5秒,  0分20秒)
  Event 3: 00:52 - 01:20  (时长:   28.0秒,  0分28秒)
  Event 4: 01:20 - 01:49  (时长:   29.4秒,  0分29秒)
  Event 5: 01:49 - 02:29  (时长:   39.7秒,  0分39秒)
  Event 6: 02:29 - 03:35  (时长:   65.4秒,  1分05秒)
  Event 7: 03:35 - 04:11  (时长:   36.4秒,  0分36秒)
  Event 8: 04:11 - 04:38  (时长:   26.8秒,  0分26秒)

  🔹 分割点 (7个): 00:31, 00:52, 01:20, 01:49, 02:29, 03:35, 04:11


🔵 ATOMIC层 - 142 个事件
  ... (省略剩余 122 个原子事件)


✅ 完成！使用与参数倒推测试完全一致的方法

💡 关键差异:
   • 片段边界: prev = b + 1（边界后从下一帧开始）
   • 特征提取: features[star

---
## 🎯 多粒度分割

In [5]:
def labels_to_segments(labels, atomic_segs):
    """将聚类标签转换为片段列表（使用原子片段的实际边界）"""
    segments = []
    K = len(labels)
    event_boundaries = [0]
    for i in range(1, K):
        if labels[i] != labels[i-1]:
            event_boundaries.append(i)
    event_boundaries.append(K)
    for i in range(len(event_boundaries) - 1):
        start_idx = event_boundaries[i]
        end_idx = event_boundaries[i+1] - 1
        start_frame = atomic_segs[start_idx][0]
        end_frame = atomic_segs[end_idx][1]
        segments.append((start_frame, end_frame))
    return segments

# 生成多粒度分割
hierarchical_segs = {}
hierarchical_segs['activity'] = labels_to_segments(fcluster(Z, t=2, criterion='maxclust'), segs)
hierarchical_segs['side'] = labels_to_segments(fcluster(Z, t=4, criterion='maxclust'), segs)
hierarchical_segs['component'] = labels_to_segments(fcluster(Z, t=8, criterion='maxclust'), segs)
hierarchical_segs['atomic'] = segs

print(f"✓ Activity层: {len(hierarchical_segs['activity'])} 个事件")
print(f"✓ Side层: {len(hierarchical_segs['side'])} 个事件")
print(f"✓ Component层: {len(hierarchical_segs['component'])} 个事件")
print(f"✓ Atomic层: {len(hierarchical_segs['atomic'])} 个事件")
print()

✓ Activity层: 2 个事件
✓ Side层: 4 个事件
✓ Component层: 8 个事件
✓ Atomic层: 142 个事件



---
## 📋 完整时间线输出

In [6]:
print("="*70)
print("📋 完整分割时间线")
print("="*70)
print()

for level_name, icon, title in zip(
    ['activity', 'side', 'component', 'atomic'],
    ['🔴', '🟠', '🟢', '🔵'],
    ['ACTIVITY', 'SIDE', 'COMPONENT', 'ATOMIC']
):
    segs_level = hierarchical_segs[level_name]
    num_events = len(segs_level)
    print(f"{icon} {title}层 - {num_events} 个事件")
    print("="*70)
    for i, (start_frame, end_frame) in enumerate(segs_level):
        start_time = start_frame / FPS
        end_time = end_frame / FPS
        duration_event = end_time - start_time
        if level_name != 'atomic' or num_events <= 20:
            print(f"  Event {i+1}: {format_time(start_time)} - {format_time(end_time)}  "
                  f"(时长: {duration_event:6.1f}秒, {int(duration_event/60):2d}分{int(duration_event%60):02d}秒)")
        elif i == 20:
            print(f"  ... (省略剩余 {num_events-20} 个原子事件)")
    if num_events > 1 and level_name != 'atomic':
        boundary_times = [seg[1] / FPS for seg in segs_level[:-1]]
        print(f"\n  🔹 分割点 ({len(boundary_times)}个): {', '.join([format_time(t) for t in boundary_times])}")
    print("\n")

print("="*70)
print("✅ 完成！使用zzzzz.ipynb保存的分割结果")
print("="*70)
print()
print("💡 说明:")
print("   • 直接加载zzzzz.ipynb中保存的分割结果")
print("   • 所有事件之间无间隙")
print()

# 验证：显示原子单元中心点的分割位置（参数倒推测试使用的方法）
print("="*70)
print("🔍 验证：原子单元中心点的分割位置（参数倒推测试方法）")
print("="*70)
print()

labels_activity = fcluster(Z, t=2, criterion='maxclust')
center_splits = []
for i in range(1, K):
    if labels_activity[i] != labels_activity[i-1]:
        split_time = int(atomic_centers[i])
        center_splits.append(split_time)
        print(f"原子单元 {i} 的中心点: {format_time(atomic_centers[i])} ({atomic_centers[i]:.1f}秒)")
        print(f"  该原子单元范围: 帧{segs[i][0]}-{segs[i][1]} = {format_time(segs[i][0]/FPS)}-{format_time(segs[i][1]/FPS)}")
        print(f"  实际边界帧: {segs[i][0]} ({format_time(segs[i][0]/FPS)})")
        print()

print(f"参数倒推测试报告的分割点（原子中心点）: {', '.join([format_time(t) for t in center_splits])}")
print(f"实际事件边界（帧边界）: {', '.join([format_time(seg[1]/FPS) for seg in hierarchical_segs['activity'][:-1]])}")
print()
print("💡 结论：参数倒推测试使用原子中心点来匹配目标，但实际分割边界在原子单元的起始帧")
print()

📋 完整分割时间线

🔴 ACTIVITY层 - 2 个事件
  Event 1: 00:00 - 02:29  (时长:  149.5秒,  2分29秒)
  Event 2: 02:29 - 04:38  (时长:  128.8秒,  2分08秒)

  🔹 分割点 (1个): 02:29


🟠 SIDE层 - 4 个事件
  Event 1: 00:00 - 01:20  (时长:   80.2秒,  1分20秒)
  Event 2: 01:20 - 02:29  (时长:   69.2秒,  1分09秒)
  Event 3: 02:29 - 03:35  (时长:   65.4秒,  1分05秒)
  Event 4: 03:35 - 04:38  (时长:   63.3秒,  1分03秒)

  🔹 分割点 (3个): 01:20, 02:29, 03:35


🟢 COMPONENT层 - 8 个事件
  Event 1: 00:00 - 00:31  (时长:   31.5秒,  0分31秒)
  Event 2: 00:31 - 00:52  (时长:   20.5秒,  0分20秒)
  Event 3: 00:52 - 01:20  (时长:   28.0秒,  0分28秒)
  Event 4: 01:20 - 01:49  (时长:   29.4秒,  0分29秒)
  Event 5: 01:49 - 02:29  (时长:   39.7秒,  0分39秒)
  Event 6: 02:29 - 03:35  (时长:   65.4秒,  1分05秒)
  Event 7: 03:35 - 04:11  (时长:   36.4秒,  0分36秒)
  Event 8: 04:11 - 04:38  (时长:   26.8秒,  0分26秒)

  🔹 分割点 (7个): 00:31, 00:52, 01:20, 01:49, 02:29, 03:35, 04:11


🔵 ATOMIC层 - 142 个事件
  ... (省略剩余 122 个原子事件)


✅ 完成！使用zzzzz.ipynb保存的分割结果

💡 说明:
   • 直接加载zzzzz.ipynb中保存的分割结果
   • 所有事件之间无间隙

🔍 验证：原子单元中心点